# 04 — Clinical adjustment with multiple imputation (MICE)

Tests whether the TME-archetype cfRNA scores (LN, FMAC, TEX) stay associated with outcome after adjustment for baseline clinical variables.
Missing clinical covariates are multiply imputed with MICE (m = 20, maxit = 20, seed 123) and model estimates are pooled with Rubin's rules.

Two analyses, each with its own imputation:

1. **Clinical adjustment (Supplementary Figure 9, Tables S3/S4).** Scores from `counts_norm` (nbin = 6), per SD.
   Baseline variables as stated in the Methods: IPI, LDH (log2), Day −30 LDH (log2), bridging therapy, stage III–IV.
   Logistic regression for response and Cox regression for PFS (stratified by cohort; PFS time = `pfs_time_fu_days`).
2. **Univariate Cox regression (Supplementary Figure 2).** Scores from the multi-source TMM-CPM matrix (as in Figure 2), per SD,
   patients infused before 2024-12-01 (`km_include`), PFS time = `pfs_time_km_days`.

**Inputs** (input_dir): `counts/counts_norm_cart_paper.tsv.gz`, `counts/counts_TMMCPM_cart_paper_ALLDATA.tsv.gz`,
`metadata/sample_metadata.tsv`, `metadata/multisource_metadata.tsv`, `reference/gencode.biotype.name.key.tsv.gz`.

**Outputs** (`results/mice/`):

| file | content | used by |
|---|---|---|
| `analysis_data.tsv` | per-patient scores (per SD), baseline covariates, PFS (analysis 1, before imputation) | SFig9 |
| `logistic_by_adjustment.csv` | OR per SD of each score: unadjusted, + each covariate, full (Table S3) | SFig9A |
| `cox_by_adjustment.csv` | same for PFS hazard ratios | — |
| `univariate_logistic.csv` | univariate OR of each score and covariate (Table S4) | SFig9B |
| `univariate_cox.csv` | univariate HR of each score and covariate | — |
| `LN_full_multivariable_logistic.csv`, `LN_full_multivariable_cox.csv` | LN + all baseline variables, all terms | — |
| `LN_LRT_incremental_value.csv` | likelihood-ratio test of adding LN to the clinical model (complete cases) | — |
| `sfig2_univariate_cox.csv` | univariate HR of each clinical variable (analysis 2) | SFig2 |
| `sfig2_scores_multivariable_cox.csv` | LN + FMAC + TEX in one Cox model (analysis 2) | SFig2 |

In [ ]:
source("../../utils/setup.R")
suppressPackageStartupMessages({
  library(mice)
  library(broom)     # tidy() methods used by mice::pool()
  library(survival)
})

## Load inputs

In [ ]:
meta      <- load_sample_metadata(order = "paper")
meta_ms   <- load_multisource_metadata()
gene_key  <- load_gene_key()
counts_norm_sym <- ensembl_to_symbol(load_counts("norm"), gene_key)
counts_tmm_all  <- load_counts("tmmcpm_all")

SIGNATURES <- c("LN_set1", "FMAC_set1", "TEX_set1")
score_sets <- function(expr, ...) {
  cbind(AddGeneSetScore(expr, LN_SYMBOLS,   name = "LN_set",   ...),
        AddGeneSetScore(expr, FMAC_SYMBOLS, name = "FMAC_set", ...),
        AddGeneSetScore(expr, TEX_SYMBOLS,  name = "TEX_set",  ...))
}

## Analysis 1 — clinical adjustment (Supplementary Figure 9, Tables S3/S4)

### Scores and baseline covariates

In [ ]:
BASELINE_VARS <- c("IPI", "LDH", "LDH_D30", "Bridging", "Stage")

scores <- score_sets(counts_norm_sym[, meta$sample_id], nbin = 6, ctrl = 100)
scores$sample_id <- rownames(scores)

df <- recode_clinical(meta, "baseline") %>%
  inner_join(scores, by = "sample_id") %>%
  transmute(sample_id, cohort, response, pfs_days, last_fu_days, timepoint,
            # per-SD scores: every OR/HR for a score is per one-SD increase
            across(all_of(SIGNATURES), ~ as.numeric(scale(.x))),
            IPI, LDH, LDH_D30, Bridging, Stage, Lymphoma_malignancy,
            pfs_event, pfs_time = pfs_time_fu_days)

cat("n =", nrow(df), "| R =", sum(df$response == "R"), "| NR =", sum(df$response == "NR"),
    "| PFS events =", sum(df$pfs_event), "\n")
sapply(df[BASELINE_VARS], function(x) sum(!is.na(x)))
write.table(df, res_path("mice", "analysis_data.tsv"), sep = "\t", quote = FALSE, row.names = FALSE)

### Multiple imputation

In [ ]:
imp_df <- df %>%
  mutate(H0 = nelsonaalen(df, pfs_time, pfs_event),   # cumulative hazard as imputation predictor
         Bridging = factor(Bridging), Stage = factor(Stage)) %>%
  select(response, cohort, all_of(SIGNATURES), all_of(BASELINE_VARS), pfs_event, H0, pfs_time)

meth <- make.method(imp_df)
meth[c("response", "cohort", SIGNATURES, "pfs_event", "H0", "pfs_time")] <- ""  # not imputed
meth[c("IPI", "LDH", "LDH_D30")] <- "pmm"
meth[c("Bridging", "Stage")]     <- "logreg"

set.seed(123)
imp <- mice(imp_df, m = 20, maxit = 20, method = meth,
            predictorMatrix = make.predictorMatrix(imp_df), printFlag = FALSE)

### Supplementary Figure 9A / Table S3 — scores adjusted for each covariate and for all

In [ ]:
ADJUSTMENTS <- c(list(Unadjusted = NULL),
                 setNames(as.list(BASELINE_VARS), paste0("+ ", BASELINE_VARS)),
                 list(`Full (all baseline, MICE)` = BASELINE_VARS))

by_adjustment <- function(fitter) {
  bind_rows(lapply(SIGNATURES, function(sig) bind_rows(lapply(names(ADJUSTMENTS), function(a)
    fitter(sig, ADJUSTMENTS[[a]], imp) %>% mutate(signature = sig, adjustment = a)))))
}
logistic_df <- by_adjustment(fit_adjusted_logistic_mi)
cox_df      <- by_adjustment(fit_adjusted_cox_mi)

write.csv(logistic_df, res_path("mice", "logistic_by_adjustment.csv"), row.names = FALSE)
write.csv(cox_df,      res_path("mice", "cox_by_adjustment.csv"),      row.names = FALSE)
print(logistic_df %>% select(signature, adjustment, n, OR, OR_lo, OR_hi, p_value), n = Inf)

### Supplementary Figure 9B / Table S4 — univariate models

In [ ]:
univariate_logistic <- bind_rows(lapply(c(SIGNATURES, BASELINE_VARS), fit_univariate_logistic_mi, imp = imp)) %>%
  mutate(label = clean_baseline_term(term))
univariate_cox <- bind_rows(lapply(c(SIGNATURES, BASELINE_VARS), fit_univariate_cox_mi, imp = imp)) %>%
  mutate(label = clean_baseline_term(term))

write.csv(univariate_logistic, res_path("mice", "univariate_logistic.csv"), row.names = FALSE)
write.csv(univariate_cox,      res_path("mice", "univariate_cox.csv"),      row.names = FALSE)
univariate_logistic
univariate_cox

### LN with all baseline variables, and its incremental value

In [ ]:
cox_fml <- function(terms) as.formula(paste0("Surv(pfs_time, pfs_event) ~ ",
                                             paste(terms, collapse = " + "), " + strata(cohort)"))

fit_log_mi <- fit_across_imputations(reformulate(c("LN_set1", BASELINE_VARS), "response"), imp,
                                     function(f, d) glm(f, data = d, family = binomial))
LN_full_log_res <- tidy_pooled_logistic(pool(fit_log_mi)) %>% mutate(model = "LN_full_MI")

fit_cox_mi <- fit_across_imputations(cox_fml(c("LN_set1", BASELINE_VARS)), imp,
                                     function(f, d) coxph(f, data = d))
LN_full_cox_res <- tidy_pooled_cox(pool(fit_cox_mi)) %>% mutate(model = "LN_full_MI")

write.csv(LN_full_log_res, res_path("mice", "LN_full_multivariable_logistic.csv"), row.names = FALSE)
write.csv(LN_full_cox_res, res_path("mice", "LN_full_multivariable_cox.csv"),      row.names = FALSE)
LN_full_log_res
LN_full_cox_res

In [ ]:
# Likelihood-ratio tests need nested models on the same rows, so they use complete cases
# (a sensitivity check; the pooled MI estimates above are the primary result).
cc_full <- df %>% filter(if_all(all_of(c("LN_set1", BASELINE_VARS, "pfs_time", "pfs_event")), ~ !is.na(.)))
cat("Complete cases: n =", nrow(cc_full), "| events =", sum(cc_full$pfs_event), "\n")

lrt_logistic <- anova(glm(reformulate(BASELINE_VARS, "response"), data = cc_full, family = binomial),
                      glm(reformulate(c("LN_set1", BASELINE_VARS), "response"), data = cc_full, family = binomial),
                      test = "LRT")
lrt_cox <- anova(coxph(cox_fml(BASELINE_VARS), data = cc_full),
                 coxph(cox_fml(c("LN_set1", BASELINE_VARS)), data = cc_full))

lrt_summary <- tibble(
  comparison = c("Clinical vs +LN (logistic)", "Clinical vs +LN (Cox)"),
  chisq      = c(lrt_logistic[2, "Deviance"], lrt_cox[2, "Chisq"]),
  df         = c(lrt_logistic[2, "Df"],       lrt_cox[2, "Df"]),
  p_value    = c(lrt_logistic[2, "Pr(>Chi)"], lrt_cox[2, "Pr(>|Chi|)"]),
  n          = nrow(cc_full)
)
write.csv(lrt_summary, res_path("mice", "LN_LRT_incremental_value.csv"), row.names = FALSE)
lrt_summary

## Analysis 2 — univariate Cox regression (Supplementary Figure 2)

Scores are computed on all 128 multi-source samples (cfRNA, PBMC, TME; default nbin = 25), exactly as for Figure 2,
then restricted to cfRNA patients with `km_include` and standardized within them.
Censored patients have PFS time 500 days (`pfs_time_km_days`), as in the Kaplan–Meier curves.

In [ ]:
CLINICAL_VARS <- c("lines_of_therapy", "IPI_prelymphodepletion", "LDH_prelymphodepletion", "stage_advanced",
                   "LDH_Day30", "De_novo_DLBCL", "Timepoint", "Bridging")

ms_scores <- score_sets(counts_tmm_all[, meta_ms$sample_id])
ms_scores$sample_id <- rownames(ms_scores)

clin <- recode_clinical(meta, "cox_mice") %>%
  select(sample_id, km_include, pfs_time_km_days, pfs_event, all_of(CLINICAL_VARS))

km_df <- meta_ms %>%
  select(sample_id, cohort) %>%                      # cohort keeps the 5 multi-source factor levels
  inner_join(ms_scores, by = "sample_id") %>%
  inner_join(clin, by = "sample_id") %>%
  filter(km_include) %>%
  mutate(LN_set1_z   = as.numeric(scale(LN_set1)),
         FMAC_set1_z = as.numeric(scale(FMAC_set1)),
         TEX_set1_z  = as.numeric(scale(TEX_set1)))
cat("n =", nrow(km_df), "| PFS events =", sum(km_df$pfs_event), "\n")

In [ ]:
imp2_df <- km_df %>%
  mutate(H0 = nelsonaalen(km_df, pfs_time_km_days, pfs_event)) %>%
  select(sample_id, LN_set1_z, FMAC_set1_z, TEX_set1_z, cohort,
         all_of(CLINICAL_VARS), pfs_time_km_days, pfs_event, H0)

meth2 <- make.method(imp2_df)
meth2[c("sample_id", "LN_set1_z", "FMAC_set1_z", "TEX_set1_z", "cohort",
        "pfs_time_km_days", "pfs_event", "H0")] <- ""
meth2[c("lines_of_therapy", "IPI_prelymphodepletion", "LDH_prelymphodepletion", "LDH_Day30")] <- "pmm"
meth2[c("stage_advanced", "De_novo_DLBCL", "Bridging", "Timepoint")] <- "logreg"  # all two-level factors
pred2 <- make.predictorMatrix(imp2_df)
pred2[, "sample_id"] <- 0
pred2["sample_id", ] <- 0

set.seed(123)
imp2 <- mice(imp2_df, m = 20, maxit = 20, method = meth2, predictorMatrix = pred2, printFlag = FALSE)

In [ ]:
sfig2_univariate <- bind_rows(lapply(CLINICAL_VARS, fit_univariate_cox_mi, imp = imp2,
                                     time = "pfs_time_km_days", event = "pfs_event",
                                     strata = NULL, col_style = "broom"))

fit_scores <- fit_across_imputations(
  Surv(pfs_time_km_days, pfs_event) ~ LN_set1_z + FMAC_set1_z + TEX_set1_z, imp2,
  function(f, d) coxph(f, data = d))
sfig2_scores <- tidy_pooled_cox(pool(fit_scores), col_style = "broom") %>%
  mutate(n = nrow(imp2$data), n_events = sum(imp2$data$pfs_event))

write.csv(sfig2_univariate, res_path("mice", "sfig2_univariate_cox.csv"), row.names = FALSE)
write.csv(sfig2_scores,     res_path("mice", "sfig2_scores_multivariable_cox.csv"), row.names = FALSE)
sfig2_univariate
sfig2_scores